In [15]:
import asyncio
import subprocess
import json
import datetime
from pathlib import Path

BASE_URL = "https://ai-test-automation-demo.onrender.com"

# ------------------ PARSE NL ------------------

def parse_test_case(nl_text):
    lines = [l.strip() for l in nl_text.split("\n") if l.strip()]
    test_name = lines[0].replace("Goal:", "").strip().lower().replace(" ", "_")

    raw_steps = []
    in_steps = False
    for line in lines:
        lower = line.lower()
        if lower.startswith("steps"):
            in_steps = True
            continue
        if lower.startswith("expected"):
            in_steps = False
            continue
        if in_steps and line[0].isdigit():
            raw_steps.append(line.split(".", 1)[1].strip())

    return {"test_name": test_name, "raw_steps": raw_steps}

# ------------------ NL → IR ------------------

def interpret_step(text):
    t = text.lower()

    if t.startswith("navigate to"):
        path = text.split("to", 1)[1].strip()
        return {"action": "navigate", "value": BASE_URL + path}

    if t.startswith("enter username"):
        value = text.split('"')[1]
        return {"action": "enter_text", "selector": "username", "value": value}

    if t.startswith("enter password"):
        value = text.split('"')[1]
        return {"action": "enter_text", "selector": "password", "value": value}

    if t.startswith("click login"):
        return {"action": "click", "selector": "login_button"}

    if t.startswith("verify dashboard_heading"):
        return {"action": "assert_visible", "selector": "dashboard_heading"}

    return {"action": "raw", "value": text}

def build_ir(parsed):
    return {
        "test_name": parsed["test_name"],
        "steps": [interpret_step(s) for s in parsed["raw_steps"]]
    }

# ------------------ SELECTOR MAP ------------------

SELECTOR_MAP = {
    "username": 'input[name="username"]',
    "password": 'input[name="password"]',
    "login_button": 'button[type="submit"]',
    "dashboard_heading": "h1",
}

def map_ir(ir):
    mapped = []
    for step in ir["steps"]:
        new_step = step.copy()
        if "selector" in new_step:
            logical = new_step["selector"]
            if logical in SELECTOR_MAP:
                new_step["selector"] = SELECTOR_MAP[logical]
        mapped.append(new_step)
    return {"test_name": ir["test_name"], "steps": mapped}

# ------------------ CODEGEN ------------------

ACTION_MAP = {
    "navigate": lambda s: f'        await page.goto("{s["value"]}")',
    "enter_text": lambda s: f"        await page.fill('{s['selector']}', '{s['value']}')",
    "click": lambda s: f"        await page.click('{s['selector']}')",
    "assert_visible": lambda s: f"        await expect(page.locator('{s['selector']}')).to_be_visible()",
}

def generate_test_code(ir):
    header = [
        "import asyncio",
        "from playwright.async_api import async_playwright, expect",
        "",
        f"async def test_{ir['test_name']}():",
        "    async with async_playwright() as p:",
        "        browser = await p.chromium.launch(headless=False)",
        "        page = await browser.new_page()",
        "",
    ]

    body = []
    for step in ir["steps"]:
        action = step.get("action")
        if action in ACTION_MAP:
            body.append(ACTION_MAP[action](step))
        else:
            body.append(f'        # Unmapped step: {step}')

    footer = [
        "",
        "        await browser.close()",
        "",
        "if __name__ == '__main__':",
        f"    asyncio.run(test_{ir['test_name']}())",
    ]

    return "\n".join(header + body + footer)

# ------------------ EXECUTION ------------------

def run_test_file(path: Path):
    result = subprocess.run(["python", str(path)], capture_output=True, text=True)

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    artifacts = Path("artifacts") / f"{path.stem}_{timestamp}"
    artifacts.mkdir(parents=True, exist_ok=True)

    summary = {
        "file": str(path),
        "return_code": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
        "artifacts": str(artifacts),
    }

    with open(artifacts / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    return summary

# ------------------ FULL PIPELINE ------------------

def run_pipeline(nl_path: str, output_path: str = "generated_test_login.py"):
    nl_text = Path(nl_path).read_text()
    parsed = parse_test_case(nl_text)
    ir = build_ir(parsed)
    mapped = map_ir(ir)
    code = generate_test_code(mapped)

    path = Path(output_path)
    path.write_text(code)

    summary = run_test_file(path)
    print("Return code:", summary["return_code"])
    print("STDOUT:\n", summary["stdout"])
    print("STDERR:\n", summary["stderr"])
    print("Artifacts:", summary["artifacts"])


In [9]:
import os
os.getcwd()


'/Users/jeffreylgoode/Documents/ai-test-automation-demo/notebooks'

In [16]:
run_pipeline("../sample-data/test_cases/test_login_success.txt")


Return code: 0
STDOUT:
 
STDERR:
 
Artifacts: artifacts/generated_test_login_20260610_090455


In [18]:
!pip install openai


  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 5.4 MB/s eta 0:00:00a 0:00:01
Using cached anyio-4.13.0-py3-none-any.whl (114 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 3.7 MB/s eta 0:00:00a 0:00:01
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
Using cached h11-0.16.0-py3-none-any.whl

In [21]:
!pip install groq



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [25]:
from groq import Groq
client = Groq(api_key="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")


In [27]:
from groq import Groq
from pathlib import Path
# paste in secret
client = Groq(api_key="secret")

nl_text = Path("../sample-data/test_cases/test_login_brittle_llm.txt").read_text()

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": "Generate a Playwright test from this natural language description. Do NOT ask questions. Do NOT request clarification. Just produce the test code."
        },
        {
            "role": "user",
            "content": nl_text
        }
    ]
)

generated_code = response.choices[0].message.content
Path("generated_brittle_test.py").write_text(generated_code)

print("Generated brittle test:")
print(generated_code)


Generated brittle test:
```python
from playwright.sync_api import sync_playwright

def test_brittle_llm_behavior():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        context = browser.new_context()
        page = context.new_page()
        page.goto("https://ai-test-automation-demo.onrender.com/login")

        # Put the username in the username box
        page.locator("#username").click()
        page.locator("#username").fill("username123")

        # Put the password in the password box
        page.locator("#password").click()
        page.locator("#password").fill("password123")

        # Hit the button to sign in
        page.locator("//button[text()='Sign in']").click()

        # Make sure the dashboard shows up
        page.wait_for_url("https://ai-test-automation-demo.onrender.com/dashboard")

        # Expected result: The user is logged in and sees the dashboard
        assert page.title() == "Dashboard"

        # Clean up
     

In [ ]:
%%sql
